## Load tables

In [ ]:
ER_PRS_F := loadCSV("data/ER_PRS_F.CSV?delimiter=,");
IR_BEN_R := loadCSV("data/IR_BEN_R.CSV?delimiter=,");
ER_PHA_F := loadCSV("data/ER_PHA_F.CSV?delimiter=,");
IR_PHA_R := loadCSV("data/IR_PHA_R.CSV?delimiter=,");

## Define VTL metadata and reduce tables

In [ ]:
ER_PRS_F_meta := ER_PRS_F	[
								calc
									identifier BEN_NIR_PSA := BEN_NIR_PSA,
									identifier BEN_RNG_GEM := BEN_RNG_GEM,
									identifier FLX_DIS_DTD := FLX_DIS_DTD,
									identifier FLX_TRT_DTD := FLX_TRT_DTD,
									identifier FLX_EMT_TYP := FLX_EMT_TYP,
									identifier FLX_EMT_NUM := FLX_EMT_NUM,
									identifier FLX_EMT_ORD := FLX_EMT_ORD,
									identifier ORG_CLE_NUM := ORG_CLE_NUM,
									identifier DCT_ORD_NUM := DCT_ORD_NUM,
									identifier PRS_ORD_NUM := PRS_ORD_NUM,
									identifier REM_TYP_AFF := REM_TYP_AFF
							]
							[
								keep EXE_SOI_DTD
							]
							[
								// a bit ugly
								filter instr(EXE_SOI_DTD, "2023:") > 0
							];
IR_BEN_R_meta := IR_BEN_R	[
								calc
									identifier BEN_NIR_PSA := BEN_NIR_PSA,
									identifier BEN_RNG_GEM := BEN_RNG_GEM
							]
							[
								keep BEN_SEX_COD, BEN_IDT_ANO
							];

ER_PHA_F_meta := ER_PHA_F	[
								calc	
									identifier FLX_DIS_DTD := FLX_DIS_DTD,
									identifier FLX_TRT_DTD := FLX_TRT_DTD,
									identifier FLX_EMT_TYP := FLX_EMT_TYP,
									identifier FLX_EMT_NUM := FLX_EMT_NUM,
									identifier FLX_EMT_ORD := FLX_EMT_ORD,
									identifier ORG_CLE_NUM := ORG_CLE_NUM,
									identifier DCT_ORD_NUM := DCT_ORD_NUM,
									identifier PRS_ORD_NUM := PRS_ORD_NUM,
									identifier REM_TYP_AFF := REM_TYP_AFF
							]
							[
								keep PHA_PRS_C13
							];


IR_PHA_R_meta := IR_PHA_R	[
								calc	
									identifier PHA_PRS_C13 := PHA_CIP_C13
							]
							[ 
								keep PHA_ATC_C03
							]
							[
								filter PHA_ATC_C03 = "J01"
							];

## Merge the 4 datasets using the correct join keys

In [ ]:
join1 := inner_join	(ER_PRS_F_meta, IR_BEN_R_meta using BEN_NIR_PSA, BEN_RNG_GEM);
join2 := inner_join	(join1, ER_PHA_F_meta 
						using 	FLX_DIS_DTD, FLX_TRT_DTD, FLX_EMT_TYP, FLX_EMT_NUM,
								FLX_EMT_ORD, ORG_CLE_NUM, DCT_ORD_NUM, PRS_ORD_NUM,
								REM_TYP_AFF
					)
					[filter not(isnull(PHA_PRS_C13))]
					[calc identifier PHA_PRS_C13 := PHA_PRS_C13];
join3 := inner_join	(join2, IR_PHA_R_meta using PHA_PRS_C13);

## Count distinct

Distinct is missing in VLT (**it will be introduced in the VTL 2.3 version**)

We need a hack:

- We need to count distinct BEN_IDT_ANO group by BEN_SEX_COD
- Grouping by BEN_SEX_COD and BEN_IDT_ANO enable to keep distinct BEN_IDT_ANO
	  We can then count by BEN_SEX_COD

In [ ]:
join3_uniq := join3[
  		aggr n := count() group by BEN_SEX_COD, BEN_IDT_ANO
	];

result <- join3_uniq	[
  		aggr nb := count() group by BEN_SEX_COD
	]
	[filter BEN_SEX_COD in {"1", "2"}]
	[rename BEN_SEX_COD to sex]
	[calc sex_label := if sex = "1" then "Man" else "Woman"];

s := show(result);